# 返修实验加速版：服务器分配38项
保持600回合、奖励50次、评估1000次、5重复和32轮S2V。当前任务清单已排除本机承担的任务与3个已有完整结果。
请把本包放入新的revision_experiments_fast_20261029目录，避免覆盖旧代码。服务器的数据仍使用上一级YX目录与其Hyper_Distance。
命令行与Notebook二选一启动，不能重复运行同一份任务。

In [ ]:
from pathlib import Path
import subprocess, sys, json
ROOT=Path.cwd().resolve()
assert (ROOT/'run_suite.py').exists(), '请在加速版目录打开Notebook'
PLAN=ROOT/'plan.json'
DATA_ROOT=ROOT.parent
CACHE_ROOT=ROOT.parent/'Hyper_Distance'
TASK_FILE=ROOT/'tasks_server.json'
OUTPUT=ROOT/'results/server_fast'
WORKERS=2
S2V_WORKERS=2  # 同时2个S2V；内存紧张时改为1。先确认服务器有足够可用内存。
def run_cli(*args):
    cmd=[sys.executable,'-u',str(ROOT/'run_suite.py'),*map(str,args)]
    process=subprocess.Popen(cmd,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True)
    status=process.wait()
    if status: raise subprocess.CalledProcessError(status,cmd)
print('Python:',sys.executable)
print('服务器分配任务数:',len(json.loads(TASK_FILE.read_text())))
print('数据:',DATA_ROOT,'缓存:',CACHE_ROOT)

## 工程测试与真实文件检查（不属于正式实验）

In [ ]:
subprocess.run([sys.executable,'-m','unittest','test_shared_experiment_core','test_portable_suite','test_fast_backend','-q'],cwd=ROOT,check=True)
run_cli('check','--plan',PLAN,'--data-root',DATA_ROOT,'--cache-root',CACHE_ROOT)

## 正式运行服务器分配任务
每个任务的细分tqdm和报错写入results/server_fast/任务编号/console.log。下面显示总进度。重跑同一命令自动跳过完成项并恢复回合检查点。

In [ ]:
run_cli('run','--plan',PLAN,'--data-root',DATA_ROOT,'--cache-root',CACHE_ROOT,
        '--task-file',TASK_FILE,'--output',OUTPUT,'--workers',WORKERS,'--s2v-workers',S2V_WORKERS)

## 查看正在运行的任务明细（可在另一个Notebook中执行）

In [ ]:
for lock in OUTPUT.glob('*/RUNNING.lock'):
    console=lock.parent/'console.log'
    print(lock.parent.name)
    if console.exists(): print(console.read_text(encoding='utf-8',errors='replace')[-2500:])

## 结束后搬回完整results/server_fast，再与本机results/local_fast合并
不要只搬CSV；原始样本、检查点和complete.json都应保留。